# Data Exploration & Audit Notebook

This notebook loads the cleaned `.parquet` data files for 2023 and 2019. 

It performs three main tasks:
1.  **Audit:** Compares the row counts of the *raw* data to the *cleaned* data to see how many rows were filtered out.
2.  **Explore Answers:** Uses the `SVDataFrame` class to inspect the schema, columns, and individual cell values for *voter/candidate answers*.
3.  **Explore Questions:** Loads and inspects the *question text* dataframes.

## 1. Setup Project Path

This cell finds the project's root folder and adds it to the system path. This is necessary so that the next cell can successfully import `from configs.base_constants import ...`

In [ ]:
import sys
import os
from pathlib import Path

# Start from the current working directory
cwd = Path.cwd()
PROJECT_ROOT = None

# Search upwards (max 5 levels) for the project root
for _ in range(5):
    # We know we're at the root if we see these folders
    if (cwd / "dependencies").is_dir() and (cwd / "vqs").is_dir():
        PROJECT_ROOT = cwd
        break
    # If not found, go one level up
    cwd = cwd.parent

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project root. Please run Jupyter from within your project folder.")

# 1. Add project root to sys.path to find 'dependencies' and 'configs'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
    print(f"Added to sys.path: {PROJECT_ROOT}")

# 2. Change CWD to root for easy file access ('data/cleaned')
os.chdir(PROJECT_ROOT)
print(f"Current Working Directory: {os.getcwd()}")

## 2. Load Dependencies & Helper Functions

Now we can import everything from our modules, including our `base_constants`.

**Note:** You must **Restart the Kernel** (circular arrow button ⟳ at the top) if you get an `ImportError` after changing `base_constants.py`.

In [ ]:
import pandas as pd
from dependencies import SVDataFrame  # This import should now work!
from pathlib import Path
from typing import Optional

# --- NO MORE DUPLICATION! ---
# We can now import *everything* from our single source of truth.
from configs.base_constants import (
    CLEANED_DIR, 
    RAW_CAND_2023_PATH,
    RAW_VOTERS_2023_PATH, 
    RAW_CAND_2019_PATH,
    RAW_VOTERS_2019_PATH,
    VOTERS_19_PREFIX,
    VOTERS_PREFIX,
    CANDIDATES_19_PREFIX,
    CANDIDATES_PREFIX,
    QUESTIONS_2023_PATH, 
    QUESTIONS_2019_PATH   
)

def load_parquet_by_prefix(directory: Path, prefix: str) -> Optional[SVDataFrame]:
    """
    Finds, loads, and wraps a parquet file in an SVDataFrame.
    """
    print(f"\nSearching for file with prefix: '{prefix}'...")
    files = list(directory.glob(f"{prefix}*.parquet"))
    
    if not files:
        print(f"❌ No file found starting with '{prefix}' in {directory}")
        return None

    file_path = files[0]
    print(f"✅ Found: {file_path.name}")
    try:
        df = pd.read_parquet(file_path)
        sv_df = SVDataFrame(df)
        print(f"Successfully loaded and wrapped in SVDataFrame.")
        return sv_df
    except Exception as e:
        print(f"❌ Error reading or wrapping file: {e}")
        return None

---

# Section A: 2023 Data

---

## 3.1. Audit 2023 Candidate Data

In [ ]:
print("--- AUDITING 2023 CANDIDATE DATA ---")

# 1. Load Raw 2023 Candidates (Before)
try:
    df_cand_raw_2023 = pd.read_csv(RAW_CAND_2023_PATH, low_memory=False)
    raw_count = len(df_cand_raw_2023)
    print(f"'BEFORE' (Raw CSV) count: {raw_count} rows")
except Exception as e:
    print(f"❌ Error loading raw file {RAW_CAND_2023_PATH}: {e}")
    raw_count = 0

# 2. Load Cleaned 2023 Candidates (After)
df_cand = load_parquet_by_prefix(CLEANED_DIR, CANDIDATES_PREFIX)
if df_cand is not None:
    cleaned_count = len(df_cand)
    print(f"'AFTER' (Cleaned Parquet) count: {cleaned_count} rows")

    # 3. Report
    if raw_count > 0:
        lost_count = raw_count - cleaned_count
        lost_percent = (lost_count / raw_count) * 100
        print(f"\nFiltered out: {lost_count} candidates ({lost_percent:.2f}%)")

## 3.2. Audit 2023 Voter Data

In [ ]:
print("--- AUDITING 2023 VOTER DATA ---")

# 1. Load Preprocessed 2023 Voters (Before)
try:
    df_voters_raw_2023 = pd.read_parquet(RAW_VOTERS_2023_PATH)
    raw_count = len(df_voters_raw_2023)
    print(f"'BEFORE' (Preprocessed Parquet) count: {raw_count} rows")
except Exception as e:
    print(f"❌ Error loading preprocessed file {RAW_VOTERS_2023_PATH}: {e}")
    raw_count = 0

# 2. Load Cleaned 2023 Voters (After)
df_voters = load_parquet_by_prefix(CLEANED_DIR, VOTERS_PREFIX)
if df_voters is not None:
    cleaned_count = len(df_voters)
    print(f"'AFTER' (Cleaned Parquet) count: {cleaned_count} rows")

    # 3. Report
    if raw_count > 0:
        lost_count = raw_count - cleaned_count
        lost_percent = (lost_count / raw_count) * 100
        print(f"\n--- 2023 VOTER DATA LOSS REPORT ---")
        print(f"Total rows filtered by clean_voters(): {lost_count}")
        print(f"Percentage of 'junk' data removed: {lost_percent:.2f}%")

In [ ]:
if df_voters_raw_2023 is not None:
    print(df_voters_raw_2023.loc[800, 'recID'])

## 3.3. Inspect 2023 Data Columns

### Candidates

In [ ]:
if df_cand is not None:
    all_columns_cand = df_cand.columns.tolist()
    
    print(f"Total columns found: {len(all_columns_cand)}")
    print("---")
    print(all_columns_cand)

In [ ]:
# Inspect a single cell value of candidates DataFrame
if df_cand is not None:
    print(df_cand.loc[0, 'ID_district'])

### Voters

In [ ]:
if df_voters is not None:
    all_columns_voters = df_voters.columns.tolist()
    
    print(f"Total columns found: {len(all_columns_voters)}")
    print("---")
    print(all_columns_voters)

In [ ]:
# Inspect row values
if df_voters is not None:
    print(df_voters.loc[3, 'districtID'])

## 3.4. Inspect 2023 Answer Matrices

In [ ]:
if df_cand is not None:
    print("--- 2023 Candidate Answer Matrix (Head) ---")
    display(df_cand.answers().head())

In [ ]:
if df_voters is not None:
    print("--- 2023 Voter Answer Matrix (Head) ---")
    display(df_voters.answers().head())

In [ ]:
# Inspect a single cell value
if 'df_voters' in locals() and df_voters is not None:
    print(df_voters.loc[15, 'recID'])

---

# Section B: 2019 Data

---

## 4.1. Audit 2019 Candidate Data

In [ ]:
print("--- AUDITING 2019 CANDIDATE DATA ---")

# 1. Load Raw 2019 Candidates (Before)
try:
    df_cand_raw_2019 = pd.read_csv(RAW_CAND_2019_PATH, low_memory=False)
    raw_count = len(df_cand_raw_2019)
    print(f"'BEFORE' (Raw CSV) count: {raw_count} rows")
except Exception as e:
    print(f"❌ Error loading raw file {RAW_CAND_2019_PATH}: {e}")
    raw_count = 0

# 2. Load Cleaned 2019 Candidates (After)
df_cand19 = load_parquet_by_prefix(CLEANED_DIR, CANDIDATES_19_PREFIX)
if df_cand19 is not None:
    cleaned_count = len(df_cand19)
    print(f"'AFTER' (Cleaned Parquet) count: {cleaned_count} rows")

    # 3. Report
    if raw_count > 0:
        lost_count = raw_count - cleaned_count
        lost_percent = (lost_count / raw_count) * 100
        print(f"\nFiltered out: {lost_count} candidates ({lost_percent:.2f}%)")

## 4.2. Audit 2019 Voter Data

In [ ]:
print("--- AUDITING 2019 VOTER DATA ---")

# 1. Load Raw 2019 Voters (Before)
try:
    df_voters_raw_2019 = pd.read_csv(RAW_VOTERS_2019_PATH, low_memory=False)
    raw_count = len(df_voters_raw_2019)
    print(f"'BEFORE' (Raw CSV) count: {raw_count} rows")
except Exception as e:
    print(f"❌ Error loading raw file {RAW_VOTERS_2019_PATH}: {e}")
    raw_count = 0

# 2. Load Cleaned 2019 Voters (After)
df_voters19 = load_parquet_by_prefix(CLEANED_DIR, VOTERS_19_PREFIX)
if df_voters19 is not None:
    cleaned_count = len(df_voters19)
    print(f"'AFTER' (Cleaned Parquet) count: {cleaned_count} rows")

    # 3. Report
    if raw_count > 0:
        lost_count = raw_count - cleaned_count
        lost_percent = (lost_count / raw_count) * 100
        print(f"\n--- 2019 VOTER DATA LOSS REPORT ---")
        print(f"Total rows filtered by clean_voters19(): {lost_count}")
        print(f"Percentage of 'junk' data removed: {lost_percent:.2f}%")

## 4.3. Inspect 2019 Candidate Columns

In [ ]:
if 'df_cand19' in locals() and df_cand19 is not None:
    all_columns_cand19 = df_cand19.columns.tolist()
    
    print(f"Total columns found: {len(all_columns_cand19)}")
    print("---")
    print(all_columns_cand19)

In [ ]:
# Inspect a single cell value
if 'df_cand19' in locals() and df_cand19 is not None:
    try:
        # Get the first answer column name automatically
        print(f"\n--- Inspecting single cell --- ")
        print(f"Random cell value at [Row 5, Column 'ID_district']: ", end="")
        print(df_cand19.loc[5, 'ID_district'])
    except Exception as e:
        print(f"Error inspecting cell: {e}")

## 4.4 Inspecting 2019 Voter Columns

In [ ]:
if 'df_voters19' in locals() and df_voters19 is not None:
    all_columns_voters19 = df_voters19.columns.tolist()
    
    print(f"Total columns found: {len(all_columns_voters19)}")
    print("---")
    print(all_columns_voters19)

In [ ]:
# Inspect a single cell value
if 'df_voters19' in locals() and df_voters19 is not None:
    try:
        # Get the first answer column name automatically
        print(f"\n--- Inspecting single cell --- ")
        print(f"Value at [Row 0, Column 'ID_district']: ", end="")
        print(df_voters19.loc[0, 'ID_district'])
    except Exception as e:
        print(f"Error inspecting cell: {e}")

## 4.4. Inspect 2019 Answer Matrices

In [ ]:
if 'df_cand19' in locals() and df_cand19 is not None:
    print("--- 2019 Candidate Answer Matrix (Head) ---")
    display(df_cand19.answers().head())

In [ ]:
if 'df_voters19' in locals() and df_voters19 is not None:
    print("--- 2019 Voter Answer Matrix (Head) ---")
    display(df_voters19.answers().head())

---

# Section C: Question Text Data

This section loads the question text dataframes we created with the `build_question_dfs.py` script. This text is the input for SBERT.

## 5.1. Load & Inspect 2023 Questions

In [ ]:
print("--- LOADING 2023 QUESTIONS ---")

try:
    df_questions = pd.read_parquet(QUESTIONS_2023_PATH)
    print(f"Shape: {df_questions.shape}")
except Exception as e:
    print(f"❌ Error loading {QUESTIONS_2023_PATH}: {e}")

In [ ]:
# Explore the 2023 questions
print("--- 2023 Question Columns ---")
print(df_questions.columns.tolist())

In [ ]:
print("\n--- 2023 Question Data (Head) ---")
display(df_questions.head())

In [ ]:
# print a sample question
qu = df_questions.get('question_EN', ['No "question_EN" column found.'])[0]
print(qu)

## 5.2. Load & Inspect 2019 Questions

Note: Rows start at index 1 for some reason.

In [ ]:
print("--- LOADING 2019 QUESTIONS ---")

try:
    df_questions19 = pd.read_parquet(QUESTIONS_2019_PATH)
    print(f"Shape: {df_questions19.shape}")
except Exception as e:
    print(f"❌ Error loading {QUESTIONS_2019_PATH}: {e}")

In [ ]:
# Explore the 2019 questions
print("--- 2019 Question Columns ---")
print(df_questions19.columns.tolist())

In [ ]:
print("\n--- 2019 Question Data (Head) ---")
display(df_questions19.head())

In [ ]:
print("\n--- Example Question (for SBERT) ---")
first_q_text_2019 = df_questions19.get('question_en')[1] # row starts from 1 for some reason
print(first_q_text_2019)